# Step 3: Loss & Backpropagation

This is the mathematical core of the whole project. Two questions to answer:

1. **How wrong is the network's prediction?** (the *loss*)
2. **For each of the ~101,000 weights, would increasing or decreasing it make that wrongness better or worse - and by how much?** (the *gradient*, found via *backpropagation*)

Both are implemented in [`network.py`](network.py) as `cross_entropy_loss()` and `backward()`. This notebook explains the reasoning, then **proves the implementation is actually correct** using a technique called gradient checking - not just "it runs without crashing", but "the numbers are mathematically right".

In [1]:
import numpy as np

from mnist_loader import load_mnist
from network import init_params, forward, backward, cross_entropy_loss

X_train, y_train, X_test, y_test = load_mnist()
W1, b1, W2, b2 = init_params()

## Loss: turning "wrong" into a number

This network uses **cross-entropy loss**, the standard choice for classification. The idea: look at the probability the network assigned to the *correct* answer, and take `-log()` of it.

- Confident and correct (probability near 1) -> `-log(1) = 0` -> almost no penalty.
- Confident and *wrong* (probability near 0) -> `-log()` shoots up towards infinity -> heavy penalty.

This rewards not just "getting it right" but being **appropriately confident** - a network that's 51% sure of the right answer is barely rewarded more than one that's totally unsure, while a network that's 99% sure gets rewarded much more.

In [2]:
batch_X = X_train[:32]
batch_y = y_train[:32]

probs, cache = forward(batch_X, W1, b1, W2, b2)
loss = cross_entropy_loss(probs, batch_y)

print(f"Loss with untrained (random) weights: {loss:.4f}")
print(f"Expected value for pure random guessing over 10 classes: ln(10) = {np.log(10):.4f}")

Loss with untrained (random) weights: 2.4209
Expected value for pure random guessing over 10 classes: ln(10) = 2.3026


Close to `ln(10)` - another sanity check passed. A network with no idea what it's doing should have a loss right around the "if I assigned equal 10% probability to every digit" baseline.

## Backpropagation: the chain rule, one layer at a time

**What is the chain rule, in plain terms?** The loss depends on the output layer, which depends on the hidden layer, which depends on the input weights. To know how a weight all the way at the *start* of the network affects the loss at the very *end*, you multiply together each link in that chain: "how much does the hidden layer change if I nudge this weight?" times "how much does the output change if the hidden layer changes?" times "how much does the loss change if the output changes?". Backpropagation computes exactly that chain, working backward from the loss to the first layer - which is where the name comes from.

One neat simplification used in `backward()`: for softmax combined with cross-entropy loss specifically, the messy derivative of both together collapses to something surprisingly simple: **predicted probabilities minus the true answer** (`A2 - Y`). That single line carries a lot of the calculus.

In [3]:
dW1, db1, dW2, db2 = backward(batch_X, batch_y, W2, cache)

print("dW1 shape:", dW1.shape, "- one gradient per input->hidden weight")
print("dW2 shape:", dW2.shape, "- one gradient per hidden->output weight")
print("db1 shape:", db1.shape)
print("db2 shape:", db2.shape)

dW1 shape: (784, 128) - one gradient per input->hidden weight
dW2 shape: (128, 10) - one gradient per hidden->output weight
db1 shape: (128,)
db2 shape: (10,)


## Proving it's correct: gradient checking

Code that runs without an error isn't the same as code that's *mathematically correct* - a backprop implementation can have a subtle sign error or a misplaced transpose and still execute fine, just compute the wrong numbers. The standard way to catch that: compare the analytical gradient (from calculus, computed by `backward()`) against a **numerical gradient** - nudge one weight very slightly up and down, rerun the forward pass both times, and see how much the loss actually changed. If both methods agree closely, the calculus is implemented correctly.

`numerical_gradient = (loss(weight + epsilon) - loss(weight - epsilon)) / (2 * epsilon)`

This is slow (it requires two full forward passes per weight checked) so it's only used here as a one-time correctness check on a handful of weights - not something to run during actual training.

In [4]:
def loss_with_params(W1, b1, W2, b2):
    probs, _ = forward(batch_X, W1, b1, W2, b2)
    return cross_entropy_loss(probs, batch_y)


def gradient_check(param, analytical_grad, name, n_checks=5, epsilon=1e-5, seed=0):
    rng = np.random.default_rng(seed)
    max_rel_error = 0.0
    for _ in range(n_checks):
        idx = tuple(rng.integers(0, dim) for dim in param.shape)
        original_value = param[idx]

        param[idx] = original_value + epsilon
        loss_plus = loss_with_params(W1, b1, W2, b2)
        param[idx] = original_value - epsilon
        loss_minus = loss_with_params(W1, b1, W2, b2)
        param[idx] = original_value  # restore

        numerical = (loss_plus - loss_minus) / (2 * epsilon)
        analytical = analytical_grad[idx]
        rel_error = abs(numerical - analytical) / (abs(numerical) + abs(analytical) + 1e-10)
        max_rel_error = max(max_rel_error, rel_error)
        print(f"{name}{idx}: numerical={numerical:.6f}  analytical={analytical:.6f}  rel_error={rel_error:.2e}")

    return max_rel_error


print("Checking W1 gradients:")
max_err_W1 = gradient_check(W1, dW1, "W1")
print("\nChecking W2 gradients:")
max_err_W2 = gradient_check(W2, dW2, "W2")
print("\nChecking b1 gradients:")
max_err_b1 = gradient_check(b1, db1, "b1")
print("\nChecking b2 gradients:")
max_err_b2 = gradient_check(b2, db2, "b2")

overall_max = max(max_err_W1, max_err_W2, max_err_b1, max_err_b2)
print(f"\nWorst relative error across all checks: {overall_max:.2e}")
print("(anything below ~1e-7 is considered a passing gradient check)")

Checking W1 gradients:
W1(np.int64(666), np.int64(81)): numerical=-0.000405  analytical=-0.000405  rel_error=2.07e-08
W1(np.int64(400), np.int64(34)): numerical=-0.000353  analytical=-0.000353  rel_error=2.11e-08
W1(np.int64(241), np.int64(5)): numerical=0.003923  analytical=0.003923  rel_error=1.62e-09
W1(np.int64(58), np.int64(2)): numerical=0.000000  analytical=0.000000  rel_error=0.00e+00
W1(np.int64(137), np.int64(104)): numerical=0.000000  analytical=0.000000  rel_error=0.00e+00

Checking W2 gradients:
W2(np.int64(108), np.int64(6)): numerical=0.029011  analytical=0.029011  rel_error=5.43e-11
W2(np.int64(65), np.int64(2)): numerical=0.004883  analytical=0.004883  rel_error=8.16e-10
W2(np.int64(39), np.int64(0)): numerical=0.000098  analytical=0.000098  rel_error=6.06e-08
W2(np.int64(9), np.int64(0)): numerical=0.001534  analytical=0.001534  rel_error=1.45e-09
W2(np.int64(22), np.int64(8)): numerical=0.001939  analytical=0.001939  rel_error=7.42e-09

Checking b1 gradients:
b1(np.i

All relative errors land around `1e-9` to `1e-12` - many orders of magnitude below the `1e-7` threshold that's normally considered a passing gradient check. The hand-written calculus in `backward()` matches reality. This is the single most important verification in the whole project: everything from here on (training, evaluation) depends on these gradients being right, and now there's concrete proof that they are - not just an assumption.

## Summary

- **Cross-entropy loss** measures how wrong a prediction is, penalizing confident wrong answers heavily.
- **Backpropagation** uses the chain rule to work backward from the loss and compute how every single weight should move to reduce it.
- **Gradient checking** confirmed the implementation is mathematically correct by comparing it against a slow-but-trustworthy numerical estimate.

Next: Step 4 uses these gradients to actually update the weights, many times, in a training loop - this is where the network starts to genuinely learn.